In [70]:
import numpy as np
import xarray as xr
from matplotlib import pyplot as plt
import pandas as pd
import matplotlib.cm as cm
from matplotlib.patches import Rectangle
from fonts_config import set_computer_modern, truncate_colormap
set_computer_modern()
import matplotlib as mpl
mpl.rcParams['axes.unicode_minus'] = False
from matplotlib.patches import Rectangle, Patch
import matplotlib.lines as mlines
import matplotlib.patheffects as path_effects

In [31]:
# ----------------------------------------------------------------------------
# Paths
# ----------------------------------------------------------------------------

rsl=xr.open_dataset("../../FesmData/GAPSLIP_Gowan2023_GrIS/output/rsl_dataset_reduced.nc")
gia=xr.open_dataset("../../FesmData/Schumacher2018_GIA_GrIS/data/schumacher2018_GR.nc")
topo = xr.open_dataset("/p/projects/megarun/ice_data/Greenland/GRL-8KM/GRL-8KM_TOPO-M17.nc")
iso=xr.open_dataset("../../FesmData/Leger2024_PaleoGris/output/paleogris_8km.nc")
bui = pd.read_excel("../../FesmData/Buizert2018/grl56971-sup-0002-supinfo.xlsx", header=10)
bui=bui.drop(0)
icec_id=["ngrip","grip","camp_century","dye3"]
icec=np.array([[75.0166666,72.5833,77.1667,65.1833326],[ -42.5333312,-37.6333,-61.1333,-43.8166634]])
rsl=xr.open_dataset("../../FesmData/GAPSLIP_Gowan2023_GrIS/output/rsl_dataset_reduced.nc")

In [3]:
def add_map(ax, topo, gia, iso, rsl):
    lgm=xr.open_dataset("/p/projects/megarun/luciagu/data/leger2024/lgm.nc")
    lgm_leg = xr.where((lgm.mask==3)|(lgm.mask==2),1,0)
    # Fondo: Morlinghem et al. 2017
    ax.set_facecolor("white")
    ax.contour(topo.xc, topo.yc, topo.z_srf, levels=np.linspace(0,10,1), colors="black", linewidths=0.5,zorder=3)
    ax.contourf(topo.xc, topo.yc, topo.z_srf.where(topo.z_srf>-10), colors="#98C37C",zorder=1)

    mask = np.where(np.isnan(topo.z_srf), 0, topo.z_srf)
    mask = np.where(mask == 0, 1, np.nan)
    
    ax.contourf(topo.xc, topo.yc, mask,colors="white",zorder=2)
    ax.contour(topo.xc, topo.yc, topo.H_ice, levels=np.linspace(100,110,1), colors="black", linewidths=0.5,zorder=3)
    ax.contourf(topo.xc, topo.yc, topo.H_ice.where(topo.H_ice>100), levels=np.linspace(0,4000,2), colors="#E6F1F3",zorder=3)

    # RSL data points
    for r_idx, reg in enumerate(rsl.region):
        sim = rsl.sel(region = reg)
        x, y = sim.xc.values, sim.yc.values
        ax.plot(x, y, '^',markersize=10,markeredgecolor='black',markerfacecolor="pink", zorder=4)
    
    # GIA
    for station in gia.station.values:
        sim=gia.sel(station=station)
        ax.plot(sim.xc, sim.yc, 'o', color="red", markersize=7, markeredgecolor='black', zorder=6)  
    
    # Ice core locations
    for (lat,lon) in zip(icec[0,:], icec[1,:]):
        dist = np.sqrt((topo.lat2D - lat)**2 + (topo.lon2D - lon)**2)
        iy, ix = np.unravel_index(dist.argmin(), dist.shape)
        x = topo.xc[ix]
        y = topo.yc[iy]
        ax.plot(x, y, '*', color="black", markersize=10, markeredgecolor='black', zorder=6)  
    ax.text(-450,-1290,"Camp Century", fontsize=12)
    ax.text(0,-1595,"NGRIP", fontsize=12)
    ax.text(150,-1830,"GRIP", fontsize=12)
    ax.text(0,-2670,"DYE 3", fontsize=12)
    
    # Isochrones
    plt.contourf(iso.xc,iso.yc,iso.age.where(iso.age>0), levels=np.linspace(0,14000,2),colors="gray", alpha=0.3, zorder=3)
    plt.contour(iso.xc,iso.yc,iso.age.where(iso.age>0), levels=np.linspace(0,14000,2),colors="black", lineswidth=1,zorder=4)
    
    # lgm
    im_lgm=ax.contour(lgm.xc, lgm.yc, lgm_leg, colors='gray',  alpha=0.1, linewidths=1.5)
    # Legend
    gia_lab = mlines.Line2D([], [],color='red',marker='o',markersize=8,markerfacecolor='red',markeredgecolor='black',linestyle='none', label="GPS stations")
    rsl_lab = mlines.Line2D([], [],color='pink',marker='^',markersize=8,markerfacecolor='pink',markeredgecolor='black',linestyle='none',label="RSL")
    lgm_lab = mlines.Line2D([], [],color='gray',markerfacecolor='gray',linestyle='solid',alpha=0.5,label="max. LGM")
    iso_lab = Rectangle((0,0), 1, 1,facecolor='gray',edgecolor='black',alpha=0.3,label='PaleoGrIS\nisochrones')
    ax.legend(handles=[gia_lab, rsl_lab, iso_lab,lgm_lab],
                loc='lower right')
    ax.set_xlim(iso.xc.min(),iso.xc.max())
    ax.set_ylim(iso.yc.min(),iso.yc.max())
    ax.tick_params(labelbottom=False, labelleft=False)
    ax.set_aspect(1)
    return


In [82]:

fig, axs = plt.subplots(2,1, figsize=(6,12),gridspec_kw={"height_ratios": [3, 10]})

ax=axs[0]
ax.plot(-bui["NGRIP_Age "].values/1e3,bui["NGRIP_ANN"].values-bui["NGRIP_ANN"].values[0],color="black")
ax.set_xlim(-22,0)
ax.set_ylim(-27,6)
ax.set_xlabel("kyr ago")
ax.set_ylabel("$\Delta T_{ann}$ (ºC)")

ax.axvspan(-22, -19, color="#deebf7", alpha=0.6)
ax.axvspan(-19, -14.8, color="white", alpha=0.8)   # azul claro
ax.axvspan(-14.6, -12.8, color="#deebf7", alpha=0.9)   # azul muy claro
ax.axvspan(-12.8, -11.5, color="#9ecae1", alpha=0.9)   # azul algo más oscuro

ax.text(-20.5, -25, "LGM", ha="center", va="center")
ax.text(-13.8, -25, "BA",  ha="center", va="center")
ax.text(-12.2, -25, "YD",  ha="center", va="center")
ax.text(-7, -25, "Holocene",  ha="center", va="center")

ax.plot([-12, -0.2], [-19, -19], '-*k', markersize=10, clip_on=False)
ax.text(-6, -17, "Surface elevations", ha="center", va="center")

ax.plot([-14, -6.5], [2, 2], color="gray", linestyle="-", linewidth=1.5)
ax.plot([-14, -6.5], [2, 2], color="gray", marker="|", markersize=12, linestyle="")
ax.plot([-19, -16], [2, 2], color="gray", linestyle="-", linewidth=1.5)
ax.plot([-19, -16], [2, 2], color="gray", marker="|", markersize=12, linestyle="")
ax.text(-13, 4, "Margin reconstructions", ha="center", va="center", color="gray")

datos_tiempo = np.array(rsl.time.where(rsl.time < 20000)).flatten()/(-1000)
datos_tiempo = datos_tiempo[~np.isnan(datos_tiempo)]
datos_tiempo_reduced = datos_tiempo[::24]
y_base = -10
ax.scatter(datos_tiempo_reduced, [y_base]*len(datos_tiempo_reduced), 
           marker='^', s=40, facecolor='pink', edgecolor='black', 
           linewidth=0.4,zorder=3)
ax.scatter(datos_tiempo.min(), y_base, 
           marker='^', s=40, facecolor='pink', edgecolor='black', 
           linewidth=0.4,zorder=3)
ax.text(-17, -10, "RSL", ha="center", va="center", color="pink",
        path_effects=[path_effects.withStroke(linewidth=0.4, foreground='black')])

ax=axs[1]
add_map(ax, topo, gia, iso, rsl)

axs[0].text(-25,2,"(a)")
axs[0].text(-25,-40,"(b)")

plt.tight_layout(h_pad=-0.2)
fig.savefig("../figs/summary_records.png", dpi=300)
plt.close()

<>:8: SyntaxWarning: invalid escape sequence '\D'
<>:8: SyntaxWarning: invalid escape sequence '\D'
/tmp/ipykernel_3099954/1477264130.py:8: SyntaxWarning: invalid escape sequence '\D'
  ax.set_ylabel("$\Delta T_{ann}$ (ºC)")
/tmp/ipykernel_3099954/4287848070.py:41: UserWarning: The following kwargs were not used by contour: 'lineswidth'
  plt.contour(iso.xc,iso.yc,iso.age.where(iso.age>0), levels=np.linspace(0,14000,2),colors="black", lineswidth=1,zorder=4)


In [66]:
datos_tiempo.min()

np.float64(nan)